# Intelligent Login Risk Analyzer  
## Notebook 07: System Testing

This notebook tests the login risk detection system using multiple simulated login scenarios.

The objective is to verify whether the system produces reasonable outputs for:
- normal login behavior
- moderately suspicious behavior
- highly suspicious behavior

## Step 1: Import Required Libraries

We import the required libraries for model loading, testing, and result comparison.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

## Step 2: Load Processed Feature Dataset

We load the processed feature dataset used to train the anomaly detection model.

In [2]:
X = pd.read_csv('../outputs/processed_features.csv')
X.head()

,is_failed,is_night_login,is_high_risk_location,is_suspicious_device,user_user10,user_user11,user_user12,user_user13,user_user14,user_user15,user_user16,user_user17,user_user18,user_user19,user_user2,user_user20,user_user3,user_user4,user_user5,user_user6,user_user7,user_user8,user_user9,location_Dhaka,location_India,location_Khulna,location_Rajshahi,location_Russia,location_Sylhet,location_UK,location_USA,device_or_browser_Edge,device_or_browser_Firefox,device_or_browser_Mobile
0,1,0,0,0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False
1,1,1,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False
2,1,1,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False
3,0,0,0,0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
4,1,0,0,1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True


## Step 3: Train the Testing Model

We retrain the Isolation Forest model so that the testing notebook remains self-contained.

In [3]:
test_model = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    random_state=42
)

test_model.fit(X)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.1
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


## Step 4: Define Helper Functions

We define the same helper functions used in the pipeline notebook so that each test scenario can be evaluated consistently.

In [4]:
def classify_risk(score):
    if score >= 6:
        return 'High'
    elif score >= 3:
        return 'Medium'
    else:
        return 'Low'


def prepare_login_features(user, time_str, location, device_or_browser, status):
    # Convert time to hour
    login_time = pd.to_datetime(time_str, format='%H:%M')
    hour = login_time.hour

    # Rule-based features
    is_failed = 1 if status == 'failed' else 0
    is_night_login = 1 if 0 <= hour <= 5 else 0
    is_high_risk_location = 1 if location == 'Russia' else 0
    is_suspicious_device = 1 if device_or_browser == 'Mobile' else 0

    # Base feature dictionary
    feature_dict = {
        'is_failed': is_failed,
        'is_night_login': is_night_login,
        'is_high_risk_location': is_high_risk_location,
        'is_suspicious_device': is_suspicious_device
    }

    # Add default values for all model columns
    for col in X.columns:
        if col not in feature_dict:
            feature_dict[col] = 0

    # One-hot encoded columns
    user_col = f'user_{user}'
    location_col = f'location_{location}'
    device_col = f'device_or_browser_{device_or_browser}'

    if user_col in feature_dict:
        feature_dict[user_col] = 1
    if location_col in feature_dict:
        feature_dict[location_col] = 1
    if device_col in feature_dict:
        feature_dict[device_col] = 1

    # Build final row
    feature_df = pd.DataFrame([feature_dict])
    feature_df = feature_df[X.columns]

    return feature_df, hour


def detect_login_risk(user, time_str, location, device_or_browser, status):
    feature_df, hour = prepare_login_features(user, time_str, location, device_or_browser, status)

    risk_score = (
        feature_df['is_failed'].iloc[0] * 3 +
        feature_df['is_night_login'].iloc[0] * 2 +
        feature_df['is_high_risk_location'].iloc[0] * 3 +
        feature_df['is_suspicious_device'].iloc[0] * 1
    )

    risk_level = classify_risk(risk_score)

    prediction = test_model.predict(feature_df)[0]
    anomaly_label = 'Anomaly' if prediction == -1 else 'Normal'

    return {
        'user': user,
        'time': time_str,
        'hour': hour,
        'location': location,
        'device_or_browser': device_or_browser,
        'status': status,
        'risk_score': risk_score,
        'risk_level': risk_level,
        'model_prediction': prediction,
        'anomaly_label': anomaly_label
    }


def should_alert(result):
    return result['risk_level'] == 'High' or result['anomaly_label'] == 'Anomaly'

## Step 5: Test Case 1 — Normal Login

This scenario represents a typical daytime successful login from a common location and browser.

In [5]:
test_1 = detect_login_risk(
    user='user1',
    time_str='10:30',
    location='Dhaka',
    device_or_browser='Chrome',
    status='success'
)

test_1

{'user': 'user1',
 'time': '10:30',
 'hour': 10,
 'location': 'Dhaka',
 'device_or_browser': 'Chrome',
 'status': 'success',
 'risk_score': np.int64(0),
 'risk_level': 'Low',
 'model_prediction': np.int64(1),
 'anomaly_label': 'Normal'}

## Step 6: Test Case 2 — Failed Login During Normal Hours

This scenario tests whether a failed login attempt during regular hours creates moderate suspicion.

In [6]:
test_2 = detect_login_risk(
    user='user2',
    time_str='11:45',
    location='Dhaka',
    device_or_browser='Edge',
    status='failed'
)

test_2

{'user': 'user2',
 'time': '11:45',
 'hour': 11,
 'location': 'Dhaka',
 'device_or_browser': 'Edge',
 'status': 'failed',
 'risk_score': np.int64(3),
 'risk_level': 'Medium',
 'model_prediction': np.int64(1),
 'anomaly_label': 'Normal'}

## Step 7: Test Case 3 — Night Login from High-Risk Location

This scenario combines late-night activity with a high-risk location.

In [7]:
test_3 = detect_login_risk(
    user='user3',
    time_str='02:20',
    location='Russia',
    device_or_browser='Chrome',
    status='success'
)

test_3

{'user': 'user3',
 'time': '02:20',
 'hour': 2,
 'location': 'Russia',
 'device_or_browser': 'Chrome',
 'status': 'success',
 'risk_score': np.int64(5),
 'risk_level': 'Medium',
 'model_prediction': np.int64(1),
 'anomaly_label': 'Normal'}

## Step 8: Test Case 4 — Highly Suspicious Login

This scenario combines several suspicious indicators:
- failed login
- night-time login
- high-risk location
- mobile device

In [8]:
test_4 = detect_login_risk(
    user='user4',
    time_str='03:10',
    location='Russia',
    device_or_browser='Mobile',
    status='failed'
)

test_4

{'user': 'user4',
 'time': '03:10',
 'hour': 3,
 'location': 'Russia',
 'device_or_browser': 'Mobile',
 'status': 'failed',
 'risk_score': np.int64(9),
 'risk_level': 'High',
 'model_prediction': np.int64(-1),
 'anomaly_label': 'Anomaly'}

## Step 9: Compare Test Results

We combine all test cases into a single table for easier comparison.

In [9]:

test_results = pd.DataFrame([test_1, test_2, test_3, test_4])
test_results

,user,time,hour,location,device_or_browser,status,risk_score,risk_level,model_prediction,anomaly_label
0,user1,10:30,10,Dhaka,Chrome,success,0,Low,1,Normal
1,user2,11:45,11,Dhaka,Edge,failed,3,Medium,1,Normal
2,user3,02:20,2,Russia,Chrome,success,5,Medium,1,Normal
3,user4,03:10,3,Russia,Mobile,failed,9,High,-1,Anomaly


## Step 10: Evaluate Alert Decisions

We check whether each scenario would trigger an alert in the system.

In [10]:
test_results['alert'] = test_results.apply(lambda row: should_alert(row), axis=1)
test_results

,user,time,hour,location,device_or_browser,status,risk_score,risk_level,model_prediction,anomaly_label,alert
0,user1,10:30,10,Dhaka,Chrome,success,0,Low,1,Normal,False
1,user2,11:45,11,Dhaka,Edge,failed,3,Medium,1,Normal,False
2,user3,02:20,2,Russia,Chrome,success,5,Medium,1,Normal,False
3,user4,03:10,3,Russia,Mobile,failed,9,High,-1,Anomaly,True


## Step 11: Testing Interpretation

A well-behaving system should show:

- low-risk or normal output for common logins
- medium suspicion for isolated failed attempts
- stronger risk for night-time or high-risk-location logins
- the strongest alerting behavior for combined suspicious conditions

This helps validate that the detection logic is aligned with cybersecurity intuition.

## Step 12: Save System Test Results

We save the testing results for use in the final demonstration notebook.

In [12]:
import os

file_path = '../outputs/system_test_results.csv'

test_results.to_csv(file_path, index=False)

if os.path.exists(file_path):
    print(f"✅ File saved successfully at: {file_path}")
else:
    print("❌ Error: File not saved")

✅ File saved successfully at: ../outputs/system_test_results.csv


## Summary

In this notebook, we:
- tested the login risk detection pipeline with multiple scenarios
- compared normal and suspicious login patterns
- verified the alerting behavior of the system
- saved structured test results for final demonstration

This notebook helps confirm that the system behaves reasonably under practical test conditions.

## Important Note

The test cases in this notebook are simulated examples designed to validate system behavior.

In a real cybersecurity environment, testing would include larger scenario coverage, historical validation, analyst feedback, and threshold tuning.